# Jiaozi Integration Update on Colab

This notebook deletes any existing `/content/Jiaozi` checkout, clones the latest `integration-update` branch, installs dependencies, and runs the integrated Jiaozi pipeline end to end on Colab.

The active flow is:

1. Module 1 parses the natural-language task with an LLM.
2. Module 2 analyzes the HuggingFace dataset.
3. Module 3 retrieves ranked CV model candidates.
4. Module 4 generates runnable training code.
5. The generated project runs the built-in smoke checks.


## Before running

1. In Colab, open **Secrets** and add `OPENAI_API_KEY`.
2. `KAGGLE_API_TOKEN` is optional here. The `integration-update` branch does not require Kaggle by default, but the notebook can store the token for future Kaggle workflows.
3. Select a GPU runtime if you want faster installs and later experimentation, though this notebook can complete on CPU.
4. Run the cells in order.


In [ ]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

def normalize_repo_url(value: str) -> str:
    value = (value or '').strip()
    markdown_match = re.fullmatch(r'\[(.*?)\]\((https?://[^)]+)\)', value)
    if markdown_match:
        return markdown_match.group(2)
    plain_url_match = re.search(r'https?://\S+', value)
    if plain_url_match:
        return plain_url_match.group(0)
    return value


REPO_URL = normalize_repo_url(os.getenv('JIAOZI_REPO_URL', 'https://github.com/Isso-W/Jiaozi.git'))
REPO_REF = os.getenv('JIAOZI_REPO_REF', 'integration-update')
REPO_DIR = Path('/content/Jiaozi')

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

print('Jiaozi repository:', REPO_DIR)
print('Jiaozi repo URL:', REPO_URL)
print('Jiaozi ref:', REPO_REF)


In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
except Exception:
    userdata = None


def read_secret(name: str, required: bool = False) -> str:
    value = ''
    if userdata is not None:
        try:
            value = userdata.get(name) or ''
        except Exception:
            value = ''
    if not value and required:
        value = getpass(f'{name} (hidden input): ').strip()
    return value.strip()


openai_api_key = read_secret('OPENAI_API_KEY', required=True)
if not openai_api_key:
    raise RuntimeError('OPENAI_API_KEY is required for this notebook.')

os.environ['OPENAI_API_KEY'] = openai_api_key
os.environ['JIAOZI_LLM_PROVIDER'] = 'openai'
os.environ['M4_LLM_PROVIDER'] = 'openai'
os.environ.setdefault('M1_OPENAI_MODEL', 'gpt-4o')
os.environ.setdefault('M4_OPENAI_MODEL', 'gpt-5.5')

kaggle_token = read_secret('KAGGLE_API_TOKEN', required=False)
if kaggle_token:
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    token_path = kaggle_dir / 'access_token'
    token_path.write_text(kaggle_token, encoding='utf-8')
    token_path.chmod(0o600)
    os.environ['KAGGLE_API_TOKEN'] = kaggle_token
    print('Stored optional KAGGLE_API_TOKEN at ~/.kaggle/access_token')
else:
    print('No KAGGLE_API_TOKEN provided. This branch does not need Kaggle for the default run.')

del openai_api_key
del kaggle_token


## Run the integrated pipeline

This sample run uses OpenAI for Module 1 parsing and `gpt-5.5` for Module 4 `model.py` generation, then lets Module 4 execute its smoke checks.


In [ ]:
import json

QUERY = 'Classify images on a small dataset with high accuracy.'
DATASET = 'uoft-cs/cifar10'
OUTPUT_DIR = Path('/content/jiaozi_generated_pipeline')

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

command = [
    sys.executable,
    'pipeline.py',
    '--query', QUERY,
    '--dataset', DATASET,
    '--fmt', 'nl',
    '--module4-output', str(OUTPUT_DIR),
    '--module4-timeout', '120',
    '--module4-llm-provider', 'openai',
]

print('Running command:', ' '.join(command))
completed = subprocess.run(command, capture_output=True, text=True)
if completed.stdout:
    print('=== pipeline stdout ===')
    print(completed.stdout)
if completed.stderr:
    print('=== pipeline stderr ===')
    print(completed.stderr)
completed.check_returncode()

summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    raise FileNotFoundError(
        f'Module 4 summary was not generated at {summary_path}. Available files: {available}'
    )


In [ ]:
summary_path = OUTPUT_DIR / 'module4_summary.json'
if not summary_path.exists():
    available = sorted(path.name for path in OUTPUT_DIR.iterdir()) if OUTPUT_DIR.exists() else []
    print(f'Module 4 summary is missing: {summary_path}')
    print('Available files:', available)
else:
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print(json.dumps(summary, indent=2, ensure_ascii=False))

print('\nGenerated files:')
if OUTPUT_DIR.exists():
    for path in sorted(OUTPUT_DIR.iterdir()):
        print(path.name)
else:
    print('Output directory was not created.')
